In [0]:
# Create widgets
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

# Retrieve widget values into variables
source_system     = dbutils.widgets.get("source_system")
table_name        = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")


In [0]:
# Build landing path
landing_path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check if path (or any parent directory) exists
try:
    path_exists = dbutils.fs.exists(landing_path)
except Exception:
    path_exists = False

if not path_exists:
    print(f"No new files received. Path does not exist: {landing_path}")
    dbutils.notebook.exit("No new files received")

In [0]:
if source_file_format == 'csv':
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp, to_timestamp

landing_df = (landing_df
    .withColumn("landing_timestamp", to_timestamp(lit(landing_timestamp), "yyyy-MM-dd_HH:mm:ss"))
    .withColumn("insert_timestamp",  current_timestamp())
)

display(landing_df)

In [0]:
landing_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")

In [0]:
%sql
select * from adb_caresync.bronze.hospitals